# Phase 5 Validation (Dimensions / Tracking)

Manual validation notebook for `pm_spark.dimensions.tracking` using `create_sample_event_log()`.

Sample columns: `case_key`, `activity`, `timestamp`, `resource`, `department`.

**Spark:** `countDistinct` is not allowed in window frames (`DISTINCT_WINDOW_FUNCTION_UNSUPPORTED`). In the library, **5.1** and **5.4** use `groupBy(case_col)` + `left join` instead.

Covered functions:
- `distinct_nonnull_count_per_case()`
- `most_frequent_value_per_case()`
- `nonnull_entry_count_per_case()`
- `dimension_stability_flag()`
- `number_of_value_changes()`
- `first_nonnull_value()`
- `last_nonnull_value()`
- `value_at_activity()`
- `multi_value_case_rate()` (returns a Python `float`, not a DataFrame)
- `null_coverage_rate()`
- `dimension_value_transition_matrix()`

In [1]:
from pathlib import Path
import sys

from pyspark.sql import SparkSession

project_root = Path.cwd().resolve()
if project_root.name == "dev":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

spark = SparkSession.getActiveSession()
if spark is None:
    spark = SparkSession.builder.appName("pm_spark_phase5_validation").getOrCreate()

import importlib

from dev.sample_data import create_sample_event_log
import pm_spark.dimensions.tracking as _dim_tracking

importlib.reload(_dim_tracking)
from pm_spark.dimensions.tracking import (
    dimension_stability_flag,
    dimension_value_transition_matrix,
    distinct_nonnull_count_per_case,
    first_nonnull_value,
    last_nonnull_value,
    most_frequent_value_per_case,
    multi_value_case_rate,
    nonnull_entry_count_per_case,
    null_coverage_rate,
    number_of_value_changes,
    value_at_activity,
)

def show_df(df, n=200):
    df.show(n, truncate=False)

26/03/22 13:25:21 WARN Utils: Your hostname, MacBook-Air-von-Alex.local resolves to a loopback address: 127.0.0.1; using 192.168.178.137 instead (on interface en0)
26/03/22 13:25:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/22 13:25:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/22 13:25:21 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/22 13:25:21 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [2]:
event_log = create_sample_event_log(spark)
show_df(event_log.orderBy("case_key", "timestamp", "activity"))

+--------+-------------+-------------------+--------+----------+
|case_key|activity     |timestamp          |resource|department|
+--------+-------------+-------------------+--------+----------+
|C001    |A            |2026-03-18 09:00:00|R1      |D1        |
|C001    |B            |2026-03-18 09:05:00|R1      |D1        |
|C001    |C            |2026-03-18 09:10:00|R1      |D1        |
|C001    |D            |2026-03-18 09:20:00|R1      |D1        |
|C002    |A            |2026-03-18 09:00:00|R1      |D1        |
|C002    |B            |2026-03-18 09:04:00|R1      |D1        |
|C002    |Manual_Review|2026-03-18 09:08:00|R1      |D1        |
|C002    |C            |2026-03-18 09:15:00|R2      |D2        |
|C002    |D            |2026-03-18 09:25:00|R2      |D2        |
|C003    |A            |2026-03-18 10:00:00|NULL    |D1        |
|C003    |C            |2026-03-18 10:10:00|R3      |NULL      |
|C003    |D            |2026-03-18 10:30:00|R3      |D3        |
|C004    |A            |2

In [3]:
# 5.1 distinct_nonnull_count_per_case() — resource (groupBy+join in library, not window)
df_d1 = distinct_nonnull_count_per_case(
    df=event_log,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    dimension_col="resource",
    output_col="resource_distinct_count",
)
show_df(df_d1.select("case_key", "resource", "resource_distinct_count").distinct().orderBy("case_key"))

# Expected: C007 -> 0 (all-null resource); C001 -> 1 (single R1)

+--------+--------+-----------------------+
|case_key|resource|resource_distinct_count|
+--------+--------+-----------------------+
|C001    |R1      |1                      |
|C002    |R1      |2                      |
|C002    |R2      |2                      |
|C003    |R3      |1                      |
|C003    |NULL    |1                      |
|C004    |R1      |2                      |
|C004    |R4      |2                      |
|C004    |NULL    |2                      |
|C005    |R2      |2                      |
|C005    |R3      |2                      |
|C006    |R5      |2                      |
|C006    |R6      |2                      |
|C006    |NULL    |2                      |
|C007    |NULL    |0                      |
+--------+--------+-----------------------+



In [4]:
# 5.2 most_frequent_value_per_case() — department (tie-break: lexicographic)
df_d2 = most_frequent_value_per_case(
    df=event_log,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    dimension_col="department",
    output_col="dept_mode",
)
show_df(df_d2.select("case_key", "department", "dept_mode").distinct().orderBy("case_key"))

+--------+----------+---------+
|case_key|department|dept_mode|
+--------+----------+---------+
|C001    |D1        |D1       |
|C002    |D1        |D1       |
|C002    |D2        |D1       |
|C003    |D1        |D1       |
|C003    |D3        |D1       |
|C003    |NULL      |D1       |
|C004    |D1        |D1       |
|C004    |D2        |D1       |
|C004    |NULL      |D1       |
|C005    |D1        |D4       |
|C005    |D4        |D4       |
|C005    |NULL      |D4       |
|C006    |D1        |D2       |
|C006    |D2        |D2       |
|C006    |NULL      |D2       |
|C007    |D5        |D5       |
|C007    |NULL      |D5       |
+--------+----------+---------+



In [5]:
# 5.3 nonnull_entry_count_per_case() — resource
df_d3 = nonnull_entry_count_per_case(
    df=event_log,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    dimension_col="resource",
    output_col="resource_nonnull_entries",
)
show_df(df_d3.select("case_key", "resource_nonnull_entries").distinct().orderBy("case_key"))

# Expected: C007 -> 0 non-null resource rows; C001 -> 4

+--------+------------------------+
|case_key|resource_nonnull_entries|
+--------+------------------------+
|C001    |4                       |
|C002    |5                       |
|C003    |2                       |
|C004    |4                       |
|C005    |5                       |
|C006    |4                       |
|C007    |0                       |
+--------+------------------------+



In [6]:
# 5.4 dimension_stability_flag() — department
df_d4 = dimension_stability_flag(
    df=event_log,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    dimension_col="department",
    output_col="dept_stable",
)
show_df(df_d4.select("case_key", "dept_stable").distinct().orderBy("case_key"))

# Expected: C001 stable (only D1); cases with multiple distinct depts -> False

+--------+-----------+
|case_key|dept_stable|
+--------+-----------+
|C001    |true       |
|C002    |false      |
|C003    |false      |
|C004    |false      |
|C005    |false      |
|C006    |false      |
|C007    |true       |
+--------+-----------+



In [7]:
# 5.5 number_of_value_changes() — department (null-stripped lag)
df_d5 = number_of_value_changes(
    df=event_log,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    dimension_col="department",
    output_col="dept_changes",
)
show_df(df_d5.select("case_key", "dept_changes").distinct().orderBy("case_key"))

+--------+------------+
|case_key|dept_changes|
+--------+------------+
|C001    |0           |
|C002    |1           |
|C003    |1           |
|C004    |1           |
|C005    |1           |
|C006    |1           |
|C007    |0           |
+--------+------------+



In [8]:
# 5.6 + 5.7 first_nonnull_value / last_nonnull_value — resource
df_d6 = first_nonnull_value(
    df=event_log,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    dimension_col="resource",
    output_col="resource_first",
)
df_d67 = last_nonnull_value(
    df=df_d6,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    dimension_col="resource",
    output_col="resource_last",
)
show_df(
    df_d67.select("case_key", "resource_first", "resource_last")
    .distinct()
    .orderBy("case_key")
)

# Expected: C003 resource_first = R3 (leading nulls skipped)

+--------+--------------+-------------+
|case_key|resource_first|resource_last|
+--------+--------------+-------------+
|C001    |R1            |R1           |
|C002    |R1            |R2           |
|C003    |R3            |R3           |
|C004    |R1            |R4           |
|C005    |R2            |R3           |
|C006    |R5            |R6           |
|C007    |NULL          |NULL         |
+--------+--------------+-------------+



In [9]:
# 5.8 value_at_activity() — department at first "B"
df_d8 = value_at_activity(
    df=event_log,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    dimension_col="department",
    target_activity="B",
    output_col="dept_at_B",
)
show_df(df_d8.select("case_key", "activity", "department", "dept_at_B").orderBy("case_key", "timestamp"))

+--------+-------------+----------+---------+
|case_key|activity     |department|dept_at_B|
+--------+-------------+----------+---------+
|C001    |A            |D1        |D1       |
|C001    |B            |D1        |D1       |
|C001    |C            |D1        |D1       |
|C001    |D            |D1        |D1       |
|C002    |A            |D1        |D1       |
|C002    |B            |D1        |D1       |
|C002    |Manual_Review|D1        |D1       |
|C002    |C            |D2        |D1       |
|C002    |D            |D2        |D1       |
|C003    |A            |D1        |NULL     |
|C003    |C            |NULL      |NULL     |
|C003    |D            |D3        |NULL     |
|C004    |A            |D1        |D1       |
|C004    |B            |D1        |D1       |
|C004    |B            |D1        |D1       |
|C004    |C            |NULL      |D1       |
|C004    |D            |D2        |D1       |
|C005    |A            |D1        |NULL     |
|C005    |B            |NULL      

In [10]:
# 5.9 multi_value_case_rate() — resource (Python float; one driver collect)
resource_multi_value_rate = multi_value_case_rate(
    df=event_log,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    dimension_col="resource",
)
print("resource_multi_value_rate:", resource_multi_value_rate)

resource_multi_value_rate: 0.5714285714285714


In [11]:
# 5.10 null_coverage_rate() — resource
df_d10 = null_coverage_rate(
    df=event_log,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    dimension_col="resource",
    per_case_null_share_col="resource_null_share_case",
    global_null_share_col="resource_null_share_global",
)
show_df(
    df_d10.select(
        "case_key",
        "resource",
        "resource_null_share_case",
        "resource_null_share_global",
    ).distinct().orderBy("case_key")
)

+--------+--------+------------------------+--------------------------+
|case_key|resource|resource_null_share_case|resource_null_share_global|
+--------+--------+------------------------+--------------------------+
|C001    |R1      |0.0                     |0.22580645161290322       |
|C002    |R1      |0.0                     |0.22580645161290322       |
|C002    |R2      |0.0                     |0.22580645161290322       |
|C003    |R3      |0.3333333333333333      |0.22580645161290322       |
|C003    |NULL    |0.3333333333333333      |0.22580645161290322       |
|C004    |R1      |0.2                     |0.22580645161290322       |
|C004    |R4      |0.2                     |0.22580645161290322       |
|C004    |NULL    |0.2                     |0.22580645161290322       |
|C005    |R2      |0.0                     |0.22580645161290322       |
|C005    |R3      |0.0                     |0.22580645161290322       |
|C006    |R5      |0.2                     |0.22580645161290322 

In [12]:
# 5.11 dimension_value_transition_matrix() — department
df_tm = dimension_value_transition_matrix(
    df=event_log,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    dimension_col="department",
    drop_self_transitions=False,
)
show_df(
    df_tm.orderBy(
        df_tm["transition_count"].desc(),
        "from_value",
        "to_value",
    )
)

df_tm_no_self = dimension_value_transition_matrix(
    df=event_log,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    dimension_col="department",
    drop_self_transitions=True,
)
show_df(
    df_tm_no_self.orderBy(
        df_tm_no_self["transition_count"].desc(),
        "from_value",
        "to_value",
    )
)

+----------+--------+----------------+
|from_value|to_value|transition_count|
+----------+--------+----------------+
|D1        |D1      |7               |
|NULL      |D1      |6               |
|D1        |NULL    |4               |
|D2        |D2      |3               |
|NULL      |D2      |2               |
|NULL      |D5      |2               |
|D4        |D4      |2               |
|NULL      |NULL    |1               |
|NULL      |D3      |1               |
|NULL      |D4      |1               |
|D1        |D2      |1               |
|D5        |NULL    |1               |
+----------+--------+----------------+

+----------+--------+----------------+
|from_value|to_value|transition_count|
+----------+--------+----------------+
|NULL      |D1      |6               |
|D1        |NULL    |4               |
|NULL      |D2      |2               |
|NULL      |D5      |2               |
|NULL      |D3      |1               |
|NULL      |D4      |1               |
|D1        |D2      |1  

## Validation checklist (Phase 5)

- [ ] 5.1 `distinct_nonnull_count_per_case()`
- [ ] 5.2 `most_frequent_value_per_case()`
- [ ] 5.3 `nonnull_entry_count_per_case()`
- [ ] 5.4 `dimension_stability_flag()`
- [ ] 5.5 `number_of_value_changes()`
- [ ] 5.6 `first_nonnull_value()`
- [ ] 5.7 `last_nonnull_value()`
- [ ] 5.8 `value_at_activity()`
- [ ] 5.9 `multi_value_case_rate()`
- [ ] 5.10 `null_coverage_rate()`
- [ ] 5.11 `dimension_value_transition_matrix()`

When satisfied, mark Phase 5 lines `[x]` in `PROGRESS.md`.